# 1. Project Title

# Supermarket Sales & Profit Analysis

## 2. Introduction

This project analyzes historical supermarket transactions to understand sales and gross-income performance across branches, products, customers, payment modes, and time periods. The analysis is built for IBM SkillsBuild Data Analytics with AI Academic Internship Program (BharatCares in association with AICTE).

## 3. Problem Statement

Identify business performance patterns from supermarket transaction data and translate them into practical business actions using data cleaning, KPI tracking, trend analysis, visualization, and recommendation generation.

## 4. Objectives

- Evaluate total sales and gross income performance
- Compare branches, product lines, customer segments, and payment modes
- Analyze monthly, daily, and hourly trends
- Generate data-backed insights and actionable recommendations

## 5. Dataset Description

- Source: Kaggle supermarket sales dataset
- File path used in this project: `data/supermarket_sales.csv`
- Key fields include branch, city, product line, quantity, total sales, gross income, date, time, and rating.

## 6. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
pd.set_option('display.max_columns', None)


## 7. Load Dataset

In [ ]:
DATA_PATH = 'data/supermarket_sales.csv'

try:
    df = pd.read_csv(DATA_PATH)
    print(f'Dataset loaded successfully from: {DATA_PATH}')
except FileNotFoundError:
    raise FileNotFoundError('Dataset not found. Please place the CSV at data/supermarket_sales.csv')


## 8. Dataset Overview

In [ ]:
print(f'Rows: {df.shape[0]}')
print(f'Columns: {df.shape[1]}')

print() 
print('Column Names:')
print(df.columns.tolist())

print()
print('Data Types:')
print(df.dtypes)

print()
print('First 5 Rows:')
display(df.head())

print()
print('Descriptive Statistics:')
display(df.describe(include='all'))


## 9. Data Cleaning

In [ ]:
clean_df = df.copy()

missing_values = clean_df.isna().sum()
duplicate_count_before = clean_df.duplicated().sum()

print('Missing values per column:')
display(missing_values.to_frame('Missing Values'))
print()
print(f'Duplicate rows before cleaning: {duplicate_count_before}')

if duplicate_count_before > 0:
    clean_df = clean_df.drop_duplicates()

clean_df['Date'] = pd.to_datetime(clean_df['Date'], errors='coerce')
clean_df['Time'] = pd.to_datetime(clean_df['Time'], format='%H:%M', errors='coerce')

numeric_cols = ['Unit price', 'Quantity', 'Tax 5%', 'Total', 'cogs', 'gross margin percentage', 'gross income', 'Rating']
for col in numeric_cols:
    clean_df[col] = pd.to_numeric(clean_df[col], errors='coerce')

clean_df['Month'] = clean_df['Date'].dt.month
clean_df['Month_Name'] = clean_df['Date'].dt.month_name()
clean_df['Day'] = clean_df['Date'].dt.day
clean_df['Day_Name'] = clean_df['Date'].dt.day_name()
clean_df['Hour'] = clean_df['Time'].dt.hour

invalid_dates = clean_df['Date'].isna().sum()
invalid_times = clean_df['Time'].isna().sum()

clean_df = clean_df.dropna(subset=['Date', 'Time'] + numeric_cols)

duplicate_count_after = clean_df.duplicated().sum()

cleaning_summary = pd.DataFrame({
    'Item': ['Original Rows', 'Rows After Cleaning', 'Duplicates Removed', 'Invalid Date Values', 'Invalid Time Values', 'Duplicates After Cleaning'],
    'Value': [len(df), len(clean_df), duplicate_count_before, invalid_dates, invalid_times, duplicate_count_after]
})

print('Cleaning Summary:')
display(cleaning_summary)

print('Cleaned data preview:')
display(clean_df.head())


## 10. Exploratory Data Analysis

In [ ]:
print('Branch distribution:')
display(clean_df['Branch'].value_counts().to_frame('Transactions'))

print('City distribution:')
display(clean_df['City'].value_counts().to_frame('Transactions'))

print('Product line distribution:')
display(clean_df['Product line'].value_counts().to_frame('Transactions'))

print('Payment method distribution:')
display(clean_df['Payment'].value_counts().to_frame('Transactions'))


## 11. KPI Analysis

In [ ]:
kpis = {
    'Total Sales': clean_df['Total'].sum(),
    'Total Gross Income': clean_df['gross income'].sum(),
    'Total Quantity Sold': clean_df['Quantity'].sum(),
    'Total Transactions': clean_df['Invoice ID'].nunique(),
    'Average Transaction Value': clean_df['Total'].mean(),
    'Average Rating': clean_df['Rating'].mean()
}

kpi_df = pd.DataFrame(kpis.items(), columns=['KPI', 'Value'])
kpi_df['Value'] = kpi_df['Value'].apply(lambda x: round(float(x), 4))
display(kpi_df)


## 12. Sales Analysis

In [ ]:
month_order = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

sales_by_branch = clean_df.groupby('Branch', as_index=False)['Total'].sum().sort_values('Total', ascending=False)
sales_by_city = clean_df.groupby('City', as_index=False)['Total'].sum().sort_values('Total', ascending=False)
sales_by_product = clean_df.groupby('Product line', as_index=False)['Total'].sum().sort_values('Total', ascending=False)
sales_by_customer_type = clean_df.groupby('Customer type', as_index=False)['Total'].sum().sort_values('Total', ascending=False)
sales_by_gender = clean_df.groupby('Gender', as_index=False)['Total'].sum().sort_values('Total', ascending=False)
sales_by_payment = clean_df.groupby('Payment', as_index=False)['Total'].sum().sort_values('Total', ascending=False)

sales_by_month = clean_df.groupby('Month_Name', as_index=False)['Total'].sum()
sales_by_month['Month_Name'] = pd.Categorical(sales_by_month['Month_Name'], categories=month_order, ordered=True)
sales_by_month = sales_by_month.sort_values('Month_Name')

sales_by_day = clean_df.groupby('Day_Name', as_index=False)['Total'].sum()
sales_by_day['Day_Name'] = pd.Categorical(sales_by_day['Day_Name'], categories=day_order, ordered=True)
sales_by_day = sales_by_day.sort_values('Day_Name')

sales_by_hour = clean_df.groupby('Hour', as_index=False)['Total'].sum().sort_values('Hour')

for title, table in [
    ('Sales by Branch', sales_by_branch),
    ('Sales by City', sales_by_city),
    ('Sales by Product Line', sales_by_product),
    ('Sales by Customer Type', sales_by_customer_type),
    ('Sales by Gender', sales_by_gender),
    ('Sales by Payment Method', sales_by_payment),
    ('Sales by Month', sales_by_month),
    ('Sales by Day', sales_by_day),
    ('Sales by Hour', sales_by_hour)
]:
    print() 
    print(title)
    display(table)


## 13. Profit/Gross Income Analysis

In [ ]:
gross_by_branch = clean_df.groupby('Branch', as_index=False)['gross income'].sum().sort_values('gross income', ascending=False)
gross_by_product = clean_df.groupby('Product line', as_index=False)['gross income'].sum().sort_values('gross income', ascending=False)
gross_by_customer_type = clean_df.groupby('Customer type', as_index=False)['gross income'].sum().sort_values('gross income', ascending=False)
gross_by_gender = clean_df.groupby('Gender', as_index=False)['gross income'].sum().sort_values('gross income', ascending=False)
gross_by_month = clean_df.groupby('Month_Name', as_index=False)['gross income'].sum()
gross_by_month['Month_Name'] = pd.Categorical(gross_by_month['Month_Name'], categories=month_order, ordered=True)
gross_by_month = gross_by_month.sort_values('Month_Name')

for title, table in [
    ('Gross Income by Branch', gross_by_branch),
    ('Gross Income by Product Line', gross_by_product),
    ('Gross Income by Customer Type', gross_by_customer_type),
    ('Gross Income by Gender', gross_by_gender),
    ('Gross Income by Month', gross_by_month)
]:
    print()
    print(title)
    display(table)


## 14. Product Category Analysis

In [ ]:
highest_sales_product = sales_by_product.loc[sales_by_product['Total'].idxmax()]
lowest_sales_product = sales_by_product.loc[sales_by_product['Total'].idxmin()]
highest_gross_product = gross_by_product.loc[gross_by_product['gross income'].idxmax()]
lowest_gross_product = gross_by_product.loc[gross_by_product['gross income'].idxmin()]

product_summary = pd.DataFrame({
    'Metric': [
        'Highest-sales product line',
        'Lowest-sales product line',
        'Highest-gross-income product line',
        'Lowest-gross-income product line'
    ],
    'Product line': [
        highest_sales_product['Product line'],
        lowest_sales_product['Product line'],
        highest_gross_product['Product line'],
        lowest_gross_product['Product line']
    ],
    'Value': [
        highest_sales_product['Total'],
        lowest_sales_product['Total'],
        highest_gross_product['gross income'],
        lowest_gross_product['gross income']
    ]
})

product_summary['Value'] = product_summary['Value'].round(4)
display(product_summary)


## 15. Branch Analysis

In [ ]:
highest_sales_branch = sales_by_branch.loc[sales_by_branch['Total'].idxmax()]
lowest_sales_branch = sales_by_branch.loc[sales_by_branch['Total'].idxmin()]
highest_gross_branch = gross_by_branch.loc[gross_by_branch['gross income'].idxmax()]
lowest_gross_branch = gross_by_branch.loc[gross_by_branch['gross income'].idxmin()]

branch_summary = pd.DataFrame({
    'Metric': [
        'Highest-sales branch',
        'Lowest-sales branch',
        'Highest-gross-income branch',
        'Lowest-gross-income branch'
    ],
    'Branch': [
        highest_sales_branch['Branch'],
        lowest_sales_branch['Branch'],
        highest_gross_branch['Branch'],
        lowest_gross_branch['Branch']
    ],
    'Value': [
        highest_sales_branch['Total'],
        lowest_sales_branch['Total'],
        highest_gross_branch['gross income'],
        lowest_gross_branch['gross income']
    ]
})
branch_summary['Value'] = branch_summary['Value'].round(4)
display(branch_summary)


## 16. Customer Analysis

In [ ]:
customer_type_analysis = clean_df.groupby('Customer type').agg(
    Sales=('Total', 'sum'),
    Transactions=('Invoice ID', 'count'),
    Avg_Transaction_Value=('Total', 'mean'),
    Gross_Income=('gross income', 'sum')
).reset_index()

gender_analysis = clean_df.groupby('Gender').agg(
    Sales=('Total', 'sum'),
    Transactions=('Invoice ID', 'count'),
    Avg_Transaction_Value=('Total', 'mean'),
    Gross_Income=('gross income', 'sum')
).reset_index()

for col in ['Sales', 'Avg_Transaction_Value', 'Gross_Income']:
    customer_type_analysis[col] = customer_type_analysis[col].round(4)
    gender_analysis[col] = gender_analysis[col].round(4)

print('Member vs Normal customer analysis')
display(customer_type_analysis)

print('Male vs Female customer analysis')
display(gender_analysis)


## 17. Payment Method Analysis

In [ ]:
payment_analysis = clean_df.groupby('Payment').agg(
    Transactions=('Invoice ID', 'count'),
    Sales=('Total', 'sum')
).reset_index()
payment_analysis['Sales_Contribution_%'] = (payment_analysis['Sales'] / payment_analysis['Sales'].sum() * 100).round(2)
payment_analysis['Sales'] = payment_analysis['Sales'].round(4)
display(payment_analysis.sort_values('Sales', ascending=False))


## 18. Time-Based Analysis

In [ ]:
monthly_sales = sales_by_month[['Month_Name', 'Total']].dropna()
daily_sales = sales_by_day[['Day_Name', 'Total']].dropna()
hourly_sales = sales_by_hour[['Hour', 'Total']].dropna()

peak_month = monthly_sales.loc[monthly_sales['Total'].idxmax()]
peak_day = daily_sales.loc[daily_sales['Total'].idxmax()]
peak_hour = hourly_sales.loc[hourly_sales['Total'].idxmax()]

print('Monthly sales trend table')
display(monthly_sales)
print('Daily sales trend table')
display(daily_sales)
print('Hourly sales trend table')
display(hourly_sales)

print(f"Peak month: {peak_month['Month_Name']} with sales {peak_month['Total']:.4f}")
print(f"Peak day: {peak_day['Day_Name']} with sales {peak_day['Total']:.4f}")
print(f"Peak hour: {int(peak_hour['Hour'])}:00 with sales {peak_hour['Total']:.4f}")


## 19. Visualizations

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(16, 26))

sns.lineplot(data=monthly_sales, x='Month_Name', y='Total', marker='o', ax=axes[0, 0])
axes[0, 0].set_title('1. Monthly Sales Trend')
axes[0, 0].set_xlabel('Month')
axes[0, 0].set_ylabel('Sales')

sns.lineplot(data=gross_by_month.dropna(), x='Month_Name', y='gross income', marker='o', ax=axes[0, 1], color='darkgreen')
axes[0, 1].set_title('2. Monthly Gross Income Trend')
axes[0, 1].set_xlabel('Month')
axes[0, 1].set_ylabel('Gross Income')

sns.barplot(data=sales_by_branch, x='Branch', y='Total', ax=axes[1, 0])
axes[1, 0].set_title('3. Sales by Branch')
axes[1, 0].set_xlabel('Branch')
axes[1, 0].set_ylabel('Sales')

sns.barplot(data=gross_by_branch, x='Branch', y='gross income', ax=axes[1, 1], color='orange')
axes[1, 1].set_title('4. Gross Income by Branch')
axes[1, 1].set_xlabel('Branch')
axes[1, 1].set_ylabel('Gross Income')

sns.barplot(data=sales_by_product, x='Total', y='Product line', ax=axes[2, 0])
axes[2, 0].set_title('5. Sales by Product Line')
axes[2, 0].set_xlabel('Sales')
axes[2, 0].set_ylabel('Product Line')

sns.barplot(data=gross_by_product, x='gross income', y='Product line', ax=axes[2, 1], color='purple')
axes[2, 1].set_title('6. Gross Income by Product Line')
axes[2, 1].set_xlabel('Gross Income')
axes[2, 1].set_ylabel('Product Line')

sns.barplot(data=sales_by_customer_type, x='Customer type', y='Total', ax=axes[3, 0])
axes[3, 0].set_title('7. Sales by Customer Type')
axes[3, 0].set_xlabel('Customer Type')
axes[3, 0].set_ylabel('Sales')

sns.barplot(data=sales_by_gender, x='Gender', y='Total', ax=axes[3, 1], color='teal')
axes[3, 1].set_title('8. Sales by Gender')
axes[3, 1].set_xlabel('Gender')
axes[3, 1].set_ylabel('Sales')

axes[4, 0].pie(payment_analysis['Transactions'], labels=payment_analysis['Payment'], autopct='%1.1f%%', startangle=90)
axes[4, 0].set_title('9. Payment Method Distribution (Transactions)')

sns.barplot(data=hourly_sales, x='Hour', y='Total', ax=axes[4, 1], color='brown')
axes[4, 1].set_title('10. Sales by Hour')
axes[4, 1].set_xlabel('Hour')
axes[4, 1].set_ylabel('Sales')

plt.tight_layout()
plt.show()


## 20. Key Insights

In [ ]:
total_sales = kpis['Total Sales']
insights = [
    f"Branch {highest_sales_branch['Branch']} generated the highest sales ({highest_sales_branch['Total']:.2f}), while Branch {lowest_sales_branch['Branch']} was lowest ({lowest_sales_branch['Total']:.2f}).",
    f"{highest_sales_product['Product line']} was the top product line by sales ({highest_sales_product['Total']:.2f}) and gross income ({highest_gross_product['gross income']:.2f}).",
    f"{lowest_sales_product['Product line']} had the lowest sales ({lowest_sales_product['Total']:.2f}) and lowest gross income ({lowest_gross_product['gross income']:.2f}).",
    f"Normal customers contributed higher sales ({customer_type_analysis.loc[customer_type_analysis['Customer type']=='Normal','Sales'].iloc[0]:.2f}) than Members ({customer_type_analysis.loc[customer_type_analysis['Customer type']=='Member','Sales'].iloc[0]:.2f}).",
    f"Ewallet led payment contribution with sales {sales_by_payment.loc[sales_by_payment['Payment']=='Ewallet','Total'].iloc[0]:.2f} ({(sales_by_payment.loc[sales_by_payment['Payment']=='Ewallet','Total'].iloc[0]/total_sales*100):.2f}% of total sales).",
    f"Sales peaked in {peak_month['Month_Name']} ({peak_month['Total']:.2f}), on {peak_day['Day_Name']} ({peak_day['Total']:.2f}), and around {int(peak_hour['Hour'])}:00 ({peak_hour['Total']:.2f}).",
    f"Average transaction value was {kpis['Average Transaction Value']:.2f} and average customer rating was {kpis['Average Rating']:.2f}."
]

for i, insight in enumerate(insights, 1):
    print(f'{i}. {insight}')


## 21. Business Recommendations

In [ ]:
recommendations = [
    (
        f"Finding: Branch {highest_sales_branch['Branch']} leads in sales and gross income.",
        f"Recommendation: Use Branch {highest_sales_branch['Branch']} as a benchmark and replicate its high-performing practices in Branch {lowest_sales_branch['Branch']} through cross-branch training and SKU planning."
    ),
    (
        f"Finding: {highest_sales_product['Product line']} is the strongest product category, while {lowest_sales_product['Product line']} is the weakest.",
        f"Recommendation: Maintain higher inventory for {highest_sales_product['Product line']} and run targeted promotions to improve movement of {lowest_sales_product['Product line']}."
    ),
    (
        'Finding: Normal customers currently spend more than Members.',
        'Recommendation: Launch a conversion campaign (discount points/loyalty onboarding) to convert high-spending Normal shoppers into Members for better retention.'
    ),
    (
        'Finding: Ewallet contributes the largest share of sales.',
        'Recommendation: Prioritize digital-payment offers with e-wallet providers while keeping cash/card options smooth for customer convenience.'
    ),
    (
        f"Finding: Peak demand occurs in {peak_month['Month_Name']}, especially on {peak_day['Day_Name']} around {int(peak_hour['Hour'])}:00.",
        'Recommendation: Increase staffing and checkout readiness during peak periods and schedule replenishment before peak windows.'
    )
]

for idx, (finding, recommendation) in enumerate(recommendations, 1):
    print(f'{idx}. {finding}')
    print(f'   {recommendation}')
    print()


## 22. Conclusion

This project cleaned and analyzed supermarket transaction data across sales, gross income, products, branches, customers, payments, and time trends. The analysis highlighted where revenue and profitability are concentrated, identified weaker areas needing action, and translated patterns into practical recommendations.

Because this dataset represents historical transactions, these findings should be used as evidence for planning and monitoring, not as a claim about current real-time supermarket behavior.